# 02 — Task 1A: Definition Extraction

Runs Prompt 1a on each final paper. For each paper, the model (Kimi K2.5) extracts every passage that uses one of the seed terms ("loss of control", "losing control", …).

**Input:** `raw_text_final/<paper_id>.txt` (the 29 final papers) · `papers.yaml` · `seed_terms_v2.yaml` · `prompts/prompt_1a_v2.txt`
**Output:** `output/<model_name>/02_task_1a/`, e.g. `output/Kimi-K2.5/02_task_1a/`

**Runs locally or in Colab:**
- **Locally** (VS Code / Jupyter): open this notebook with the project's `.venv` kernel and set the `HF_TOKEN` environment variable.
- **Colab**: add two secrets (key icon): `HF_TOKEN` and `GITHUB_TOKEN` (the repo is private). The last cell pushes the results to GitHub, because Colab deletes its files when the session ends.

**Start with `DRY_RUN = True`** (fake model replies, no token, no cost; results go to `output/<model_name>_dryrun/`). Then set `DRY_RUN = False` for the real run.


**Prerequisites:** Run `01_extract_text.ipynb` first so `raw_text/{paper_id}.txt` files exist on Drive.

In [ ]:
# Setup: works locally and in Google Colab
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_SLUG = "AI-Loc-Research/loss-of-control-definition-v2"

if IN_COLAB:
    from google.colab import userdata

    REPO_DIR = "/content/loss-of-control-definition-v2"
    github_token = userdata.get("GITHUB_TOKEN")  # private repo: a token with read (and write, to push) access

    def git(*args, cwd=None):
        result = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
        if result.returncode != 0:  # never print the token
            raise RuntimeError("git failed: " + result.stderr.replace(github_token, "***"))
        return result.stdout

    if os.path.exists(REPO_DIR):
        git("pull", cwd=REPO_DIR)
    else:
        git("clone", f"https://{github_token}@github.com/{REPO_SLUG}.git", REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"],
                   check=True)
else:
    # Locally this notebook lives in <repo>/notebooks
    REPO_DIR = str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f"Repo: {REPO_DIR}   (Colab: {IN_COLAB})")


In [ ]:
# Hugging Face token: never type it into the notebook
if IN_COLAB:
    HF_TOKEN = userdata.get("HF_TOKEN")        # Colab: key icon -> secret named HF_TOKEN
else:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")  # Local: set the HF_TOKEN environment variable

print("HF token found" if HF_TOKEN else "No HF token: only DRY_RUN = True will work")


In [ ]:
# Run settings: the paper's settings for Task 1A (Kimi K2.5, prompt v2, seed terms v2)
DRY_RUN = True   # True: fake model replies, no token, no cost. Set to False for the real run.

config = {
    "experiment": "1a_kimi-k2.5_v2",   # free-text label, saved with the run
    "model": "moonshotai/Kimi-K2.5",   # results go to output/Kimi-K2.5/02_task_1a/
    "provider": "auto",                # Hugging Face picks a live provider for this model
    "temperature": 0.0,
    "prompt_version": "prompt_1a_v2",  # prompts/prompt_1a_v2.txt (paper Appendix C)
    "seed_terms_version": "v2",        # seed_terms_v2.yaml (the paper's 8 seed terms)
    "max_new_tokens": 16384,
}


In [ ]:
# Run Task 1A. Papers that already have output are skipped, so re-running resumes.
from src.run_stage import run_1a

run_1a(config, HF_TOKEN, repo_root=REPO_DIR, dry_run=DRY_RUN)


In [ ]:
# Review docs (review/*.md), summary.csv and consolidated.json, with quote verification
from src.generate_review import generate_review_docs

generate_review_docs(config, repo_root=REPO_DIR, dry_run=DRY_RUN)


In [ ]:
# Summary table: passages found and quote verification per paper
import pandas as pd
from src.paths import task_output_dir

OUT_DIR = task_output_dir(REPO_DIR, config["model"], "02_task_1a", dry_run=DRY_RUN)
print("Results in:", OUT_DIR)
pd.read_csv(os.path.join(OUT_DIR, "summary.csv"))


In [ ]:
# Colab only: push the results to GitHub (Colab deletes its files when the session ends).
# Locally, commit and push with git as usual instead.
GIT_NAME = "Your Name"            # who the commit is from
GIT_EMAIL = "you@example.com"

if IN_COLAB and not DRY_RUN:
    results = os.path.relpath(OUT_DIR, REPO_DIR)
    git("config", "user.name", GIT_NAME, cwd=REPO_DIR)
    git("config", "user.email", GIT_EMAIL, cwd=REPO_DIR)
    git("add", results, cwd=REPO_DIR)
    git("commit", "-m", f"results: task 1A, {config['model']}", cwd=REPO_DIR)
    git("pull", "--rebase", cwd=REPO_DIR)
    git("push", cwd=REPO_DIR)
    print("Pushed", results)
else:
    print("Nothing pushed (dry run, or running locally).")
